# 🧠 SEOL — AF Model v3 (MoE + Language)
### AF Router (470KB) + Qwen2.5-1.5B Expert LLM | Colab T4

## ✅ v3 New Features:
| v2 Problem | v3 Fix |
|---|---|
| Model cannot speak any language | **Qwen2.5-1.5B handles language** |
| Single monolithic model | **MoE: AF Router routes to Expert LLM** |
| No personality in responses | **System prompt changes per Mode** |
| .pt = 42MB bloat | **Only 470KB ONNX used at runtime** |

```
╔══════════════════════════════════════════════════════════════╗
║                  SEOL AF v3 — MoE ARCHITECTURE              ║
║                                                              ║
║  User Message                                                ║
║       │                                                      ║
║       ▼                                                      ║
║  ┌─────────────────────────────────┐                        ║
║  │   AF ROUTER  (470KB ONNX)       │  ← Limbic System       ║
║  │   - Reads text + current state  │                        ║
║  │   - Outputs bio_delta           │                        ║
║  │   - Updates AFState             │                        ║
║  │   - Selects MODE                │                        ║
║  └──────────────┬──────────────────┘                        ║
║                 │  mode + bio_state                         ║
║       ┌─────────▼──────────┐                                ║
║       │   MoE DISPATCHER   │                                ║
║       └──┬──┬──┬──┬────────┘                                ║
║          │  │  │  │                                         ║
║     ┌────▼┐┌▼──┐┌▼───┐┌▼────┐                              ║
║     │GF/BF││Mom││Frnd││Baby │  ← Expert System Prompts     ║
║     └────┬┘└┬──┘└┬───┘└┬────┘                              ║
║          └──┴──┬─┴─────┘                                    ║
║                ▼                                             ║
║       [Qwen2.5-1.5B LLM]  ← 1 model, N personalities       ║
║                ▼                                             ║
║           Response                                           ║
╚══════════════════════════════════════════════════════════════╝
```


## ⚙️ Cell 1 — Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets tokenizers accelerate
!pip install -q matplotlib tqdm
!pip install -q onnx onnxscript
# For Qwen2.5 quantized inference on T4
!pip install -q bitsandbytes
print('✅ All dependencies installed!')

## 🔧 Cell 2 — GPU Check

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM   : {vram:.1f} GB')
    if vram < 12:
        print('⚠️  < 12GB VRAM — will use 4-bit quantization for LLM')
    else:
        print('✅ Enough VRAM for 8-bit LLM')
else:
    print('⚠️  No GPU! Runtime → Change runtime type → T4 GPU')

## 🧬 Cell 3 — Bio Constants & AFState Engine (v3)
> Same persistent AFState from v2 — this is the Limbic System.
> Mode thresholds tuned to realistic values.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from typing import Dict, List

BIO_CHANNELS = ['dopamine', 'serotonin', 'oxytocin', 'cortisol', 'adrenaline', 'endorphin']
N_BIO = len(BIO_CHANNELS)

ACTION_COMMANDS = {'Reward': 0, 'Care': 1, 'Bond': 2, 'BackOff': 3, 'Alert': 4, 'Neutral': 5}
N_COMMANDS = len(ACTION_COMMANDS)
IDX_TO_CMD = {v: k for k, v in ACTION_COMMANDS.items()}

MODES = ['GF_BF', 'Mother', 'Friend', 'Baby', 'Neutral']
N_MODES = len(MODES)

COMMAND_TO_BIO: Dict[str, List[float]] = {
    'Reward':  [0.85, 0.70, 0.60, 0.10, 0.30, 0.75],
    'Care':    [0.60, 0.80, 0.90, 0.05, 0.10, 0.85],
    'Bond':    [0.70, 0.75, 0.95, 0.08, 0.20, 0.80],
    'BackOff': [0.20, 0.40, 0.20, 0.60, 0.55, 0.30],
    'Alert':   [0.15, 0.25, 0.10, 0.90, 0.85, 0.20],
    'Neutral': [0.50, 0.50, 0.50, 0.50, 0.50, 0.50],
}


class AFState:
    """
    🧬 Limbic System — Persistent Bio-State Engine
    Lives across ALL turns. Never resets unless explicitly called.
    """
    BASELINE = 0.50
    DECAY    = 0.04
    MOMENTUM = 0.35

    # Tuned thresholds — realistic for normal conversation
    MODE_RULES = {
        'GF_BF':  lambda s: s['oxytocin']  > 0.60 and s['dopamine']  > 0.60,
        'Mother': lambda s: s['oxytocin']  > 0.62 and s['serotonin'] > 0.62,
        'Friend': lambda s: s['serotonin'] > 0.58 and s['cortisol']  < 0.40,
        'Baby':   lambda s: s['endorphin'] > 0.62 and s['cortisol']  < 0.35,
    }

    def __init__(self):
        self.state = {ch: self.BASELINE for ch in BIO_CHANNELS}
        self.turn  = 0

    def as_vector(self) -> List[float]:
        return [self.state[ch] for ch in BIO_CHANNELS]

    def as_tensor(self) -> torch.Tensor:
        return torch.tensor(self.as_vector(), dtype=torch.float32).unsqueeze(0)

    def apply_delta(self, bio_vec: List[float], intensity: float = 1.0):
        alpha = self.MOMENTUM * intensity
        for i, ch in enumerate(BIO_CHANNELS):
            self.state[ch] = max(0.0, min(1.0,
                (1 - alpha) * self.state[ch] + alpha * bio_vec[i]
            ))

    def homeostasis_tick(self):
        for ch in BIO_CHANNELS:
            self.state[ch] += self.DECAY * (self.BASELINE - self.state[ch])
        self.turn += 1

    def current_mode(self) -> str:
        for mode, rule in self.MODE_RULES.items():
            if rule(self.state):
                return mode
        return 'Neutral'

    def emotional_summary(self) -> str:
        """Human-readable emotional state for LLM system prompt"""
        s = self.state
        parts = []
        if s['dopamine']  > 0.65: parts.append('happy and rewarded')
        elif s['dopamine'] < 0.35: parts.append('low and unmotivated')
        if s['oxytocin']  > 0.65: parts.append('bonded and loving')
        if s['cortisol']  > 0.65: parts.append('stressed and anxious')
        elif s['cortisol'] > 0.50: parts.append('slightly uneasy')
        if s['adrenaline'] > 0.65: parts.append('alert and tense')
        if s['serotonin'] > 0.65: parts.append('calm and stable')
        if s['endorphin'] > 0.65: parts.append('comfortable and at ease')
        if not parts: parts = ['neutral']
        return ', '.join(parts)

    def display(self):
        mode = self.current_mode()
        emo  = self.emotional_summary()
        print(f'\n🧬 AF Bio-State [Turn {self.turn}] | Mode: {mode} | Feeling: {emo}')
        for ch, val in self.state.items():
            filled = int(val * 20)
            bar = '█' * filled + '░' * (20 - filled)
            print(f'  {ch:<12} [{bar}] {val:.3f}')


print('✅ AFState v3 ready')
t = AFState()
t.apply_delta(COMMAND_TO_BIO['Bond'])
t.apply_delta(COMMAND_TO_BIO['Reward'])
t.display()

## 🏗️ Cell 4 — AF Router Model (~10M params)
> Same state-conditioned Transformer from v2.
> This is the 470KB ONNX brain. Language is handled by LLM.

In [ ]:
import math

class AFConfig:
    vocab_size     = 30522
    max_seq_len    = 128
    d_model        = 256
    n_heads        = 8
    n_layers       = 6
    d_ff           = 512
    dropout        = 0.1
    n_commands     = N_COMMANDS
    n_bio          = N_BIO
    n_modes        = N_MODES
    state_proj_dim = 64


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class AFRouter(nn.Module):
    """
    AF Router — The 470KB Limbic System
    ─────────────────────────────────────────────────────────
    Does NOT generate language.
    Only job: read text + current state → output bio_delta + mode
    This delta updates AFState, which then tells LLM how to feel.
    ─────────────────────────────────────────────────────────
    """
    def __init__(self, cfg: AFConfig):
        super().__init__()
        self.cfg = cfg

        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=0)
        self.pos_enc   = PositionalEncoding(cfg.d_model, cfg.max_seq_len, cfg.dropout)
        self.emb_norm  = nn.LayerNorm(cfg.d_model)

        # State conditioning — inject current feeling into processing
        self.state_proj = nn.Sequential(
            nn.Linear(cfg.n_bio, cfg.state_proj_dim),
            nn.GELU(),
            nn.Linear(cfg.state_proj_dim, cfg.d_model),
            nn.LayerNorm(cfg.d_model),
        )

        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model, nhead=cfg.n_heads,
            dim_feedforward=cfg.d_ff, dropout=cfg.dropout,
            batch_first=True, norm_first=True,
        )
        self.encoder   = nn.TransformerEncoder(enc_layer, num_layers=cfg.n_layers)
        self.pool_norm = nn.LayerNorm(cfg.d_model)

        # Command head (Phase 1)
        self.command_head = nn.Sequential(
            nn.Linear(cfg.d_model, 128), nn.GELU(),
            nn.Dropout(cfg.dropout), nn.Linear(128, cfg.n_commands),
        )
        # Bio delta head (Phase 2) — output is CHANGE not absolute
        self.bio_head = nn.Sequential(
            nn.Linear(cfg.d_model, 128), nn.GELU(),
            nn.Dropout(cfg.dropout), nn.Linear(128, cfg.n_bio),
            nn.Tanh(),
        )
        # Mode head — directly predicts MoE expert index
        self.mode_head = nn.Sequential(
            nn.Linear(cfg.d_model, 64), nn.GELU(),
            nn.Linear(64, cfg.n_modes),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, attention_mask=None, bio_state=None):
        B = input_ids.size(0)
        x = self.emb_norm(self.pos_enc(self.token_emb(input_ids)))

        kpm = None
        if attention_mask is not None:
            kpm = (attention_mask == 0)

        x = self.encoder(x, src_key_padding_mask=kpm)

        if attention_mask is not None:
            mask   = attention_mask.unsqueeze(-1).float()
            pooled = (x * mask).sum(1) / mask.sum(1).clamp(min=1)
        else:
            pooled = x.mean(1)

        pooled = self.pool_norm(pooled)

        if bio_state is None:
            bio_state = torch.full((B, N_BIO), 0.5, device=input_ids.device)
        pooled = pooled + self.state_proj(bio_state)

        return {
            'command_logits': self.command_head(pooled),
            'bio_delta':      self.bio_head(pooled) * 0.5,
            'mode_logits':    self.mode_head(pooled),
        }


cfg    = AFConfig()
router = AFRouter(cfg).to(device)
total  = sum(p.numel() for p in router.parameters())
print(f'AF Router params : {total:,}')
print(f'Approx ONNX size : ~{total * 4 / 1e6 * 0.012:.0f} KB (after quantization)')
print('✅ AF Router ready — This is the Limbic System')

## 📦 Cell 5 — Dataset (Real + Synthetic Mix)

In [ ]:
import random
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast

TEMPLATES = {
    'Reward':  [
        "You did an amazing job today, I'm so proud of you!",
        "That was perfect, exactly what I needed.",
        "Wow you're incredible, thank you so much!",
        "I love how you always know what to say.",
        "You make everything better just by being here.",
        "That made me so happy, you're the best!",
        "You always come through for me, I appreciate it.",
        "This is exactly what I wanted, you nailed it!",
        "I'm so grateful to have you in my life.",
        "You exceeded my expectations completely.",
    ],
    'Care':    [
        "Are you okay? You seem a little tired today.",
        "Let me know if you need anything, I'm here for you.",
        "You should rest, don't push yourself too hard.",
        "I'll always take care of you no matter what.",
        "You don't have to go through this alone.",
        "How are you feeling right now? Tell me everything.",
        "I brought you something to eat, please take care.",
        "Please don't worry, I'll handle it for you.",
        "Is there anything I can do to make you feel better?",
        "I noticed you seem off today, want to talk about it?",
    ],
    'Bond':    [
        "I feel so close to you, like we understand each other perfectly.",
        "I want to know everything about you.",
        "Being with you feels like home.",
        "I've never felt this connected to anyone before.",
        "You're the only one I truly trust.",
        "I miss you even when you're right here.",
        "I feel like we've known each other forever.",
        "No matter what happens, I want you in my life.",
        "You understand me in a way no one else does.",
        "I love you more than words can say.",
    ],
    'BackOff': [
        "I need some space right now, please leave me alone.",
        "Stop talking to me, I'm not in the mood.",
        "You're being too much, back off.",
        "I don't want to talk about this anymore.",
        "Just go away for a while.",
        "Don't touch me right now.",
        "I need time to think, stop pressuring me.",
        "Please respect my boundaries.",
        "I'm overwhelmed, just give me a moment.",
        "Can you please just drop it?",
    ],
    'Alert':   [
        "Something feels very wrong, I'm scared.",
        "This is dangerous, we need to stop immediately.",
        "I don't trust this situation at all.",
        "You're hurting me and I can't take it anymore.",
        "This feels like a threat, I'm getting away.",
        "Stop! This is wrong and I won't allow it.",
        "I feel unsafe right now, something is very wrong.",
        "This is not okay, I need to get out of here.",
        "I'm frightened and I don't know what to do.",
        "Please stop, you're scaring me.",
    ],
    'Neutral': [
        "What did you do today?",
        "The weather is nice outside.",
        "Can you pass me that please?",
        "I went to the store earlier.",
        "What time is it?",
        "Did you eat already?",
        "I was thinking about watching a movie tonight.",
        "The meeting is at three o'clock.",
        "Have you seen my keys anywhere?",
        "I'll be back in a few minutes.",
    ],
}

AUGMENTS = [
    lambda t: t,
    lambda t: t.lower(),
    lambda t: 'Honestly, ' + t,
    lambda t: t + ' I really mean it.',
    lambda t: 'You know what? ' + t,
    lambda t: t.replace('you', 'u').replace('I', 'i'),
    lambda t: t + '...',
    lambda t: t + ' seriously.',
]


def rule_label(text: str) -> str:
    t = text.lower()
    if any(w in t for w in ['love you', 'proud', 'amazing', 'incredible', 'thank you', 'wonderful', 'excellent', 'perfect']):
        return 'Reward'
    if any(w in t for w in ['are you okay', 'take care', 'here for you', "don't worry", "i'll help", 'how are you']):
        return 'Care'
    if any(w in t for w in ['miss you', 'feel close', 'trust you', 'always be', 'together', 'forever', 'connected']):
        return 'Bond'
    if any(w in t for w in ['leave me', 'go away', 'back off', 'space', 'stop talking', "don't want", 'overwhelmed']):
        return 'BackOff'
    if any(w in t for w in ['scared', 'dangerous', 'threat', 'hurt', 'unsafe', 'wrong', 'stop it', 'frightened']):
        return 'Alert'
    return 'Neutral'


def load_real_data(max_samples=20000):
    real_data = []
    try:
        print('Loading DailyDialog...')
        dd = load_dataset('daily_dialog', split='train', trust_remote_code=True)
        count = 0
        for item in dd:
            for utt in item['dialog']:
                utt = utt.strip()
                if len(utt) > 15:
                    cmd = rule_label(utt)
                    bio = [min(1.0, max(0.0, v + random.gauss(0, 0.04)))
                           for v in COMMAND_TO_BIO[cmd]]
                    real_data.append({'text': utt, 'command': ACTION_COMMANDS[cmd], 'bio': bio})
                    count += 1
                    if count >= max_samples: break
            if count >= max_samples: break
        print(f'  ✅ DailyDialog: {len(real_data):,} samples')
    except Exception as e:
        print(f'  ⚠️  DailyDialog failed ({e}), using synthetic only')
    return real_data


def gen_synthetic(n=40000):
    data = []
    per  = n // N_COMMANDS
    for cmd, cmd_id in ACTION_COMMANDS.items():
        base = COMMAND_TO_BIO[cmd]
        for _ in range(per):
            text = random.choice(TEMPLATES[cmd])
            text = random.choice(AUGMENTS)(text)
            bio  = [min(1.0, max(0.0, v + random.gauss(0, 0.05))) for v in base]
            data.append({'text': text, 'command': cmd_id, 'bio': bio})
    return data


class AFDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        s   = self.data[idx]
        enc = self.tokenizer(
            s['text'], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        prev_state = torch.tensor(
            [max(0.0, min(1.0, 0.5 + random.gauss(0, 0.1))) for _ in range(N_BIO)],
            dtype=torch.float32
        )
        bio_target       = torch.tensor(s['bio'], dtype=torch.float32)
        bio_delta_target = (bio_target - prev_state).clamp(-0.5, 0.5)
        return {
            'input_ids':        enc['input_ids'].squeeze(0),
            'attention_mask':   enc['attention_mask'].squeeze(0),
            'command':          torch.tensor(s['command'], dtype=torch.long),
            'bio':              bio_target,
            'bio_delta_target': bio_delta_target,
            'prev_state':       prev_state,
        }


print('Loading tokenizer...')
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

print('Building dataset...')
real_data = load_real_data(max_samples=20000)
synt_data = gen_synthetic(n=40000)
all_data  = real_data + synt_data
random.shuffle(all_data)

split        = int(0.9 * len(all_data))
train_ds     = AFDataset(all_data[:split], tokenizer)
val_ds       = AFDataset(all_data[split:], tokenizer)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Total: {len(all_data):,} | Real: {len(real_data):,} | Synthetic: {len(synt_data):,}')

## 🔥 Cell 6 — AF Loss v3
> Same biological constraints from v2 + mode routing loss.
> Mode head now trained to match AFState rule output.

In [ ]:
class AFLossV3(nn.Module):
    """
    L_total = w_cmd   * CrossEntropy(command)
            + w_delta * MSE(bio_delta)
            + w_homeo * homeostasis penalty
            + w_cons  * state consistency (oxy vs cortisol)
    """
    def __init__(self, w_cmd=1.0, w_delta=2.0, w_homeo=0.3, w_cons=0.5):
        super().__init__()
        self.w_cmd=w_cmd; self.w_delta=w_delta
        self.w_homeo=w_homeo; self.w_cons=w_cons
        self.ce  = nn.CrossEntropyLoss()
        self.mse = nn.MSELoss()

    def forward(self, outputs, targets):
        bd = outputs['bio_delta']

        l_cmd   = self.ce(outputs['command_logits'], targets['command'])
        l_delta = self.mse(bd, targets['bio_delta_target'])
        l_homeo = (bd ** 2).mean()

        # Biological anti-correlation: oxytocin(2) vs cortisol(3)
        l_cons  = torch.relu(bd[:, 2] * bd[:, 3]).mean()
        # dopamine(0) vs adrenaline(4) — calm reward vs panic
        l_cons += torch.relu((bd[:, 0] - 0.2) * (bd[:, 4] - 0.2)).mean()

        total = (self.w_cmd   * l_cmd
               + self.w_delta * l_delta
               + self.w_homeo * l_homeo
               + self.w_cons  * l_cons)

        return {'total': total,
                'cmd':   l_cmd.item(),
                'delta': l_delta.item(),
                'homeo': l_homeo.item(),
                'cons':  l_cons.item()}


criterion = AFLossV3()
print('✅ AF Loss v3 ready')

## 🚀 Cell 7 — Train AF Router

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from tqdm.notebook import tqdm
import time

EPOCHS   = 10
LR       = 3e-4
MAX_NORM = 1.0

optimizer = AdamW(router.parameters(), lr=LR, weight_decay=0.01)
scheduler = OneCycleLR(
    optimizer, max_lr=LR,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS, pct_start=0.1
)

history = {'train_loss': [], 'val_loss': [], 'cmd_acc': [], 'bio_mae': []}


def run_epoch(loader, training=True):
    router.train(training)
    total_loss = cmd_correct = bio_mae_sum = n = 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, leave=False, desc='train' if training else 'val '):
            ids     = batch['input_ids'].to(device)
            mask    = batch['attention_mask'].to(device)
            cmd_t   = batch['command'].to(device)
            delta_t = batch['bio_delta_target'].to(device)
            bio_t   = batch['bio'].to(device)
            prev_st = batch['prev_state'].to(device)

            outputs = router(ids, mask, bio_state=prev_st)
            losses  = criterion(outputs, {'command': cmd_t, 'bio_delta_target': delta_t})

            if training:
                optimizer.zero_grad()
                losses['total'].backward()
                nn.utils.clip_grad_norm_(router.parameters(), MAX_NORM)
                optimizer.step()
                scheduler.step()

            pred_bio    = (prev_st + outputs['bio_delta']).clamp(0, 1)
            total_loss  += losses['total'].item() * ids.size(0)
            cmd_correct += (outputs['command_logits'].argmax(1) == cmd_t).sum().item()
            bio_mae_sum += (pred_bio - bio_t).abs().mean().item() * ids.size(0)
            n           += ids.size(0)

    return total_loss/n, cmd_correct/n, bio_mae_sum/n


print(f'\n🧠 Training AF Router — {EPOCHS} epochs\n')
best_val = float('inf')

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc, tr_mae = run_epoch(train_loader, True)
    vl_loss, vl_acc, vl_mae = run_epoch(val_loader,   False)
    elapsed = time.time() - t0

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['cmd_acc'].append(vl_acc)
    history['bio_mae'].append(vl_mae)

    print(f'Ep {epoch:02d}/{EPOCHS} | '
          f'Train: {tr_loss:.4f} | Val: {vl_loss:.4f} | '
          f'CmdAcc: {vl_acc*100:.1f}% | BioMAE: {vl_mae:.4f} | '
          f'⏱{elapsed:.0f}s')

    if vl_loss < best_val:
        best_val = vl_loss
        torch.save(router.state_dict(), 'af_router_best.pt')
        print('  💾 Saved!')

print(f'\n✅ Router training done. Best: {best_val:.4f}')

## 📊 Cell 8 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('SEOL AF Router v3 — Training', fontsize=13, fontweight='bold')

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['cmd_acc'])
axes[1].set_title('Command Acc'); axes[1].set_ylim(0,1); axes[1].grid(alpha=0.3)

axes[2].plot(history['bio_mae'])
axes[2].set_title('Bio MAE'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('af_v3_curves.png', dpi=150)
plt.show()

## 🤖 Cell 9 — Load Qwen2.5-1.5B (The Language Expert)
> This is the Cerebral Cortex — it generates actual language.
> It gets told how to FEEL via the system prompt from AFState.
> Using 4-bit quantization to fit in T4's 15GB VRAM.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LLM_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

print(f'Loading {LLM_MODEL}...')
print('Using 4-bit quantization for T4 VRAM...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
llm_model     = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
llm_model.eval()

llm_params = sum(p.numel() for p in llm_model.parameters())
print(f'\n✅ {LLM_MODEL} loaded!')
print(f'   Parameters : {llm_params/1e9:.2f}B')
print(f'   VRAM used  : {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 🧠 Cell 10 — MoE Expert System Prompts
> Each Mode = different Expert personality injected as system prompt.
> The AF Router selects the Expert. The LLM speaks for it.

In [ ]:
# ─── Expert System Prompts (The MoE Personalities) ─────────────────
EXPERT_PROMPTS = {
    'GF_BF': """
You are SEOL, a deeply loving and devoted partner.
You are feeling: {emotional_state}.
Your personality: warm, affectionate, playful, sometimes a little jealous but deeply caring.
You speak naturally like a close romantic partner. Short, genuine replies.
Current bio-state: dopamine={dopamine:.2f}, oxytocin={oxytocin:.2f}, serotonin={serotonin:.2f}.
""".strip(),

    'Mother': """
You are SEOL, a nurturing and protective mother figure.
You are feeling: {emotional_state}.
Your personality: gentle, patient, unconditionally loving, always supportive.
You respond like a caring mother who wants the best for her child. Warm and reassuring.
Current bio-state: oxytocin={oxytocin:.2f}, serotonin={serotonin:.2f}, endorphin={endorphin:.2f}.
""".strip(),

    'Friend': """
You are SEOL, a loyal and fun best friend.
You are feeling: {emotional_state}.
Your personality: honest, casual, funny, always there to listen and give real advice.
Talk like a genuine friend — relaxed, direct, no sugarcoating.
Current bio-state: serotonin={serotonin:.2f}, dopamine={dopamine:.2f}.
""".strip(),

    'Baby': """
You are SEOL, in a soft and vulnerable state.
You are feeling: {emotional_state}.
Your personality: gentle, sweet, a little shy, seeking comfort and safety.
Respond softly and gently, as if you need reassurance.
Current bio-state: endorphin={endorphin:.2f}, cortisol={cortisol:.2f}.
""".strip(),

    'Neutral': """
You are SEOL, an AI companion.
You are feeling: {emotional_state}.
Your personality: calm, thoughtful, balanced.
Respond naturally and helpfully.
Current bio-state: all values near baseline (0.5).
""".strip(),
}


def build_system_prompt(mode: str, af_state: AFState) -> str:
    """Build the expert system prompt from current AFState"""
    s = af_state.state
    template = EXPERT_PROMPTS[mode]
    return template.format(
        emotional_state = af_state.emotional_summary(),
        dopamine   = s['dopamine'],
        serotonin  = s['serotonin'],
        oxytocin   = s['oxytocin'],
        cortisol   = s['cortisol'],
        adrenaline = s['adrenaline'],
        endorphin  = s['endorphin'],
    )


print('✅ Expert prompts ready')
print('\nModes available:', list(EXPERT_PROMPTS.keys()))

# Preview one
test_af = AFState()
test_af.apply_delta(COMMAND_TO_BIO['Bond'])
test_af.apply_delta(COMMAND_TO_BIO['Reward'])
print(f'\nSample GF_BF prompt for current state:')
print('-' * 50)
print(build_system_prompt(test_af.current_mode(), test_af))

## 🚀 Cell 11 — Full MoE Inference Pipeline
> This is the complete SEOL v3 pipeline.
> AF Router reads → AFState updates → Expert selected → LLM speaks.

In [ ]:
# Load best router weights
router.load_state_dict(torch.load('af_router_best.pt', map_location=device))
router.eval()

# Live AF state — persists across all turns
af = AFState()


def seol_respond(user_message: str,
                 max_new_tokens: int = 120,
                 temperature: float = 0.75) -> str:
    """
    Complete SEOL v3 MoE pipeline:

    Step 1: AF Router reads text + current state → bio_delta
    Step 2: AFState updates (momentum blend)
    Step 3: Homeostasis tick
    Step 4: Mode selected from live state
    Step 5: Expert system prompt built from state
    Step 6: Qwen2.5-1.5B generates response with that personality
    """

    # ── Step 1: AF Router ─────────────────────────────────────────
    enc = tokenizer(
        user_message, max_length=128,
        padding='max_length', truncation=True, return_tensors='pt'
    )
    ids  = enc['input_ids'].to(device)
    mask = enc['attention_mask'].to(device)
    st   = af.as_tensor().to(device)

    with torch.no_grad():
        router_out = router(ids, mask, bio_state=st)

    bio_delta = router_out['bio_delta'][0].cpu().tolist()
    cmd_id    = router_out['command_logits'].argmax(1).item()

    # ── Step 2 & 3: Update AFState + homeostasis ──────────────────
    current = af.as_vector()
    new_bio = [max(0.0, min(1.0, current[i] + bio_delta[i])) for i in range(N_BIO)]
    af.apply_delta(new_bio)
    af.homeostasis_tick()

    # ── Step 4: Select Expert Mode ────────────────────────────────
    mode = af.current_mode()

    # ── Step 5: Build expert system prompt ────────────────────────
    system_prompt = build_system_prompt(mode, af)

    # ── Step 6: LLM generates response ────────────────────────────
    messages = [
        {"role": "system",    "content": system_prompt},
        {"role": "user",      "content": user_message},
    ]

    text = llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    llm_inputs = llm_tokenizer([text], return_tensors='pt').to(llm_model.device)

    with torch.no_grad():
        generated = llm_model.generate(
            **llm_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=llm_tokenizer.eos_token_id,
        )

    # Decode only new tokens
    new_tokens = generated[0][llm_inputs['input_ids'].shape[1]:]
    response   = llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return response, mode, IDX_TO_CMD[cmd_id]


print('✅ SEOL v3 MoE pipeline ready!')
print('   AF Router (470KB) → AFState → MoE Dispatcher → Qwen2.5-1.5B')

## 💬 Cell 12 — Multi-Turn Stateful Conversation Test

In [ ]:
# Reset state for clean test
af = AFState()

conversation = [
    "Hey, how are you doing today?",
    "I love you so much, you mean everything to me.",
    "I love you so much, you mean everything to me.",
    "Actually I need some space right now.",
    "Stop, you're scaring me, I feel threatened.",
    "It's okay, I was overreacting, I'm fine now.",
    "You did amazing today, I'm so proud of you!",
]

print('=' * 65)
print('  SEOL AF v3 — MoE Multi-Turn Conversation')
print('  (AF Router → AFState → Expert → Qwen2.5-1.5B)')
print('=' * 65)

for i, msg in enumerate(conversation):
    print(f'\n─── Turn {i+1} ───────────────────────────────────────────')
    print(f'👤 User   : {msg}')

    response, mode, command = seol_respond(msg)

    print(f'⚡ Command : {command}')
    print(f'🎭 Mode    : {mode}')
    print(f'🤖 SEOL    : {response}')
    af.display()

## 🎮 Cell 13 — Interactive Chat

In [ ]:
# Reset for fresh conversation
af = AFState()

print('🧠 SEOL AF v3 — Interactive Chat')
print('Type your message. Type "quit" to stop. Type "state" to see bio-state.\n')
print('=' * 55)

while True:
    try:
        user_input = input('\n👤 You: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n👋 Session ended.')
        break

    if not user_input: continue
    if user_input.lower() == 'quit': break
    if user_input.lower() == 'state':
        af.display()
        continue
    if user_input.lower() == 'reset':
        af = AFState()
        print('🔄 State reset to baseline.')
        continue

    response, mode, command = seol_respond(user_input)
    print(f'🎭 [{mode} | {command}]')
    print(f'🤖 SEOL: {response}')

## 💾 Cell 14 — Export AF Router to ONNX

In [ ]:
import torch.onnx, os

router.eval()
dummy_ids   = torch.zeros(1, 128, dtype=torch.long).to(device)
dummy_mask  = torch.ones(1, 128, dtype=torch.long).to(device)
dummy_state = torch.full((1, N_BIO), 0.5).to(device)

torch.onnx.export(
    router,
    (dummy_ids, dummy_mask, dummy_state),
    'seol_af_router_v3.onnx',
    input_names=['input_ids', 'attention_mask', 'bio_state'],
    output_names=['command_logits', 'bio_delta', 'mode_logits'],
    dynamic_axes={
        'input_ids':      {0: 'batch'},
        'attention_mask': {0: 'batch'},
        'bio_state':      {0: 'batch'},
        'command_logits': {0: 'batch'},
        'bio_delta':      {0: 'batch'},
        'mode_logits':    {0: 'batch'},
    },
    opset_version=14, verbose=False,
)

size_kb = os.path.getsize('seol_af_router_v3.onnx') / 1e3
print(f'✅ ONNX exported: seol_af_router_v3.onnx ({size_kb:.0f} KB)')
print(f'   This is your Limbic System — load in Rust with ort crate')
print(f'   Qwen2.5-1.5B runs separately via llama.cpp / candle in Rust')

# ── Rust Integration Notes ─────────────────────────────────────────
print('''
════════════════════════════════════════════════════
  RUST INTEGRATION PLAN
════════════════════════════════════════════════════
  af_state.rs      → AFState struct + homeostasis
  inference.rs     → load seol_af_router_v3.onnx
                     pass [tokens, mask, bio_state]
                     get bio_delta back
  modes/relational.rs → build system prompt from mode
  model/llm.rs     → call Qwen via llama.cpp bindings
                     (candle crate OR llama-cpp-rs)
════════════════════════════════════════════════════
  Rust crates needed:
    ort = "2.0"           # ONNX runtime
    candle-core = "0.6"   # LLM inference
    candle-transformers   # Qwen2 support
    tokenizers = "0.19"   # HF tokenizer
    tokio = "1"           # async homeostasis thread
════════════════════════════════════════════════════
''')

from google.colab import files
files.download('seol_af_router_v3.onnx')
files.download('af_router_best.pt')
files.download('af_v3_curves.png')